# Notebook 03: Baseline Benchmarking - Establishing Performance Metrics

## 📚 Historical Context

**Timeline**: 2020-2021 - Rise of rigorous inference benchmarking

**The Problem**:
- Early papers claimed "2x speedup!" without proper methodology
- No standard benchmarks (different hardware, different models, cherry-picked tests)
- GPT-3 (2020) demanded reproducible performance claims
- "Works on my machine" - but which machine?

**The Solution**:
- Standardized metrics: P50, P95, P99 latencies
- Multiple runs for statistical significance
- Proper warmup and measurement protocols
- Document hardware, batch size, sequence length
- Baseline vs optimized comparisons

## 🎯 What You'll Learn

1. Setting up proper benchmarks
2. Metrics: latency (P50/P95/P99), throughput, memory usage
3. Naive implementation baseline
4. Benchmarking framework setup
5. Statistical significance and multiple runs
6. Results visualization

## 💡 Why This Matters

- **Prove improvements**: "Trust but verify" - measure, don't guess
- **Catch regressions**: Know when changes make things worse
- **Compare fairly**: Apples-to-apples comparisons
- **Production readiness**: P99 latency matters for real users

**Key Principle**: "In God we trust, all others must bring data" - W. Edwards Deming

**Reference**: See LLM_INFERENCE_ROADMAP.md - Stage 0: Baseline Benchmarking

---

## Setup and Imports

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("=" * 80)
print("BENCHMARKING ENVIRONMENT")
print("=" * 80)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory: {props.total_memory / 1e9:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")
print("=" * 80)

## Part 1: Understanding Key Metrics

### Essential Metrics for Inference:

#### 1. **Latency Metrics**:
- **Mean**: Average latency (useful but can be misleading)
- **Median (P50)**: 50% of requests are faster than this
- **P95**: 95% of requests are faster than this
- **P99**: 99% of requests are faster than this (tail latency)
- **Why P95/P99?**: Production systems care about worst-case, not just average

#### 2. **Throughput Metrics**:
- **Tokens/second**: Total tokens generated per second
- **Requests/second**: How many requests can be handled
- **Effective throughput**: Accounting for batching overhead

#### 3. **Memory Metrics**:
- **Peak memory**: Maximum GPU memory used
- **Memory per token**: How memory grows with sequence length
- **Memory bandwidth utilization**: % of theoretical max

#### 4. **Quality Metrics** (for optimization validation):
- **Output correctness**: Does optimization change outputs?
- **Numerical precision**: FP16 vs FP32 differences

### Why Different Percentiles Matter:

```
Example latencies: [10, 11, 10, 12, 11, 10, 50, 11, 10, 12] ms

Mean:   13.7 ms  ← Misleading! Skewed by outlier
P50:    11.0 ms  ← Typical user experience
P95:    50.0 ms  ← 1 in 20 users sees this
P99:    50.0 ms  ← Worst-case latency

Production insight: If P99 is 10x P50, you have a problem!
```

In [ ]:
# Define a dataclass for benchmark results
@dataclass
class BenchmarkResult:
    """Container for benchmark results."""
    # Configuration
    model_name: str
    device: str
    batch_size: int
    sequence_length: int
    num_runs: int
    
    # Latency metrics (milliseconds)
    latency_mean: float
    latency_std: float
    latency_min: float
    latency_max: float
    latency_p50: float
    latency_p95: float
    latency_p99: float
    
    # Throughput
    throughput_tokens_per_sec: float
    
    # Memory (megabytes)
    memory_allocated_mb: Optional[float] = None
    memory_reserved_mb: Optional[float] = None
    
    def to_dict(self):
        return asdict(self)
    
    def __str__(self):
        return f"""
Benchmark Result:
  Configuration:
    Model: {self.model_name}
    Device: {self.device}
    Batch size: {self.batch_size}
    Sequence length: {self.sequence_length}
    Runs: {self.num_runs}
  
  Latency:
    Mean:  {self.latency_mean:.2f} ± {self.latency_std:.2f} ms
    P50:   {self.latency_p50:.2f} ms
    P95:   {self.latency_p95:.2f} ms
    P99:   {self.latency_p99:.2f} ms
    Range: [{self.latency_min:.2f}, {self.latency_max:.2f}] ms
  
  Throughput:
    {self.throughput_tokens_per_sec:.2f} tokens/sec
  
  Memory:
    Allocated: {self.memory_allocated_mb:.2f} MB
    Reserved:  {self.memory_reserved_mb:.2f} MB
"""

# Test the dataclass
example_result = BenchmarkResult(
    model_name="gpt2",
    device="cuda",
    batch_size=1,
    sequence_length=128,
    num_runs=20,
    latency_mean=15.5,
    latency_std=2.1,
    latency_min=13.2,
    latency_max=22.1,
    latency_p50=15.2,
    latency_p95=19.8,
    latency_p99=21.5,
    throughput_tokens_per_sec=64.5,
    memory_allocated_mb=250.5,
    memory_reserved_mb=300.0,
)

print(example_result)

## Part 2: Building a Benchmarking Framework

In [ ]:
class InferenceBenchmark:
    """
    Comprehensive benchmarking framework for LLM inference.
    """
    
    def __init__(self, model, tokenizer, device: str):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.model.eval()
    
    def warmup(self, input_ids: torch.Tensor, num_warmup: int = 10):
        """
        Warmup the model before benchmarking.
        
        Why warmup?
        - GPU needs to initialize kernels
        - CUDA caching effects
        - First runs are always slower
        """
        for _ in range(num_warmup):
            with torch.no_grad():
                _ = self.model(input_ids)
        
        if self.device == "cuda":
            torch.cuda.synchronize()
    
    def benchmark_single_forward(
        self,
        input_ids: torch.Tensor,
        num_runs: int = 50,
        num_warmup: int = 10
    ) -> BenchmarkResult:
        """
        Benchmark a single forward pass.
        """
        # Warmup
        self.warmup(input_ids, num_warmup)
        
        # Reset memory stats
        if self.device == "cuda":
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.empty_cache()
        
        # Benchmark
        latencies = []
        
        for _ in range(num_runs):
            if self.device == "cuda":
                start = torch.cuda.Event(enable_timing=True)
                end = torch.cuda.Event(enable_timing=True)
                
                start.record()
                with torch.no_grad():
                    _ = self.model(input_ids)
                end.record()
                
                torch.cuda.synchronize()
                elapsed = start.elapsed_time(end)
            else:
                start = time.time()
                with torch.no_grad():
                    _ = self.model(input_ids)
                elapsed = (time.time() - start) * 1000
            
            latencies.append(elapsed)
        
        # Calculate metrics
        latencies = np.array(latencies)
        
        # Memory metrics
        if self.device == "cuda":
            memory_allocated = torch.cuda.max_memory_allocated() / (1024**2)
            memory_reserved = torch.cuda.max_memory_reserved() / (1024**2)
        else:
            memory_allocated = None
            memory_reserved = None
        
        return BenchmarkResult(
            model_name=self.model.config.model_type,
            device=self.device,
            batch_size=input_ids.shape[0],
            sequence_length=input_ids.shape[1],
            num_runs=num_runs,
            latency_mean=float(np.mean(latencies)),
            latency_std=float(np.std(latencies)),
            latency_min=float(np.min(latencies)),
            latency_max=float(np.max(latencies)),
            latency_p50=float(np.percentile(latencies, 50)),
            latency_p95=float(np.percentile(latencies, 95)),
            latency_p99=float(np.percentile(latencies, 99)),
            throughput_tokens_per_sec=float(input_ids.shape[1] / (np.mean(latencies) / 1000)),
            memory_allocated_mb=float(memory_allocated) if memory_allocated else None,
            memory_reserved_mb=float(memory_reserved) if memory_reserved else None,
        )
    
    def benchmark_generation(
        self,
        prompt: str,
        max_new_tokens: int = 20,
        num_runs: int = 10,
        num_warmup: int = 3
    ) -> Dict[str, any]:
        """
        Benchmark full generation (prefill + decode).
        """
        input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(self.device)
        
        # Warmup
        for _ in range(num_warmup):
            generated = input_ids.clone()
            for _ in range(5):  # Short warmup generation
                with torch.no_grad():
                    outputs = self.model(generated)
                    next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
                    generated = torch.cat([generated, next_token], dim=-1)
        
        if self.device == "cuda":
            torch.cuda.synchronize()
        
        # Benchmark generation
        all_prefill_times = []
        all_decode_times = []
        all_total_times = []
        
        for _ in range(num_runs):
            generated = input_ids.clone()
            
            # Measure prefill
            if self.device == "cuda":
                start = torch.cuda.Event(enable_timing=True)
                end = torch.cuda.Event(enable_timing=True)
                start.record()
            else:
                start_time = time.time()
            
            with torch.no_grad():
                outputs = self.model(generated)
                next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
                generated = torch.cat([generated, next_token], dim=-1)
            
            if self.device == "cuda":
                end.record()
                torch.cuda.synchronize()
                prefill_time = start.elapsed_time(end)
            else:
                prefill_time = (time.time() - start_time) * 1000
            
            all_prefill_times.append(prefill_time)
            
            # Measure decode
            decode_times = []
            for _ in range(max_new_tokens - 1):
                if self.device == "cuda":
                    start = torch.cuda.Event(enable_timing=True)
                    end = torch.cuda.Event(enable_timing=True)
                    start.record()
                else:
                    start_time = time.time()
                
                with torch.no_grad():
                    outputs = self.model(generated)
                    next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
                    generated = torch.cat([generated, next_token], dim=-1)
                
                if self.device == "cuda":
                    end.record()
                    torch.cuda.synchronize()
                    elapsed = start.elapsed_time(end)
                else:
                    elapsed = (time.time() - start_time) * 1000
                
                decode_times.append(elapsed)
            
            all_decode_times.extend(decode_times)
            all_total_times.append(prefill_time + sum(decode_times))
        
        return {
            'prefill': {
                'mean': np.mean(all_prefill_times),
                'p50': np.percentile(all_prefill_times, 50),
                'p95': np.percentile(all_prefill_times, 95),
                'p99': np.percentile(all_prefill_times, 99),
            },
            'decode': {
                'mean': np.mean(all_decode_times),
                'p50': np.percentile(all_decode_times, 50),
                'p95': np.percentile(all_decode_times, 95),
                'p99': np.percentile(all_decode_times, 99),
            },
            'total': {
                'mean': np.mean(all_total_times),
                'p50': np.percentile(all_total_times, 50),
                'p95': np.percentile(all_total_times, 95),
                'p99': np.percentile(all_total_times, 99),
            },
            'throughput_tokens_per_sec': max_new_tokens / (np.mean(all_total_times) / 1000),
        }

print("Benchmarking framework created!")

## Part 3: Establishing Baseline Performance

In [ ]:
# Load model
model_name = "gpt2"

print("Loading model for baseline benchmarking...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Create benchmark object
benchmark = InferenceBenchmark(model, tokenizer, device)

In [ ]:
# Benchmark different sequence lengths
print("=" * 80)
print("BASELINE BENCHMARK: SINGLE FORWARD PASS")
print("=" * 80)

sequence_lengths = [32, 64, 128, 256, 512]
baseline_results = []

for seq_len in sequence_lengths:
    print(f"\nBenchmarking sequence length: {seq_len}")
    input_ids = torch.randint(0, tokenizer.vocab_size, (1, seq_len)).to(device)
    
    result = benchmark.benchmark_single_forward(input_ids, num_runs=50, num_warmup=10)
    baseline_results.append(result)
    
    print(f"  Mean latency: {result.latency_mean:.2f} ± {result.latency_std:.2f} ms")
    print(f"  P95 latency: {result.latency_p95:.2f} ms")
    print(f"  Throughput: {result.throughput_tokens_per_sec:.2f} tokens/sec")
    if result.memory_allocated_mb:
        print(f"  Memory: {result.memory_allocated_mb:.2f} MB")

print("\n" + "=" * 80)

## Part 4: Comprehensive Visualization

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

seq_lens = [r.sequence_length for r in baseline_results]

# Plot 1: Latency comparison (Mean, P95, P99)
ax = axes[0, 0]
means = [r.latency_mean for r in baseline_results]
p95s = [r.latency_p95 for r in baseline_results]
p99s = [r.latency_p99 for r in baseline_results]

ax.plot(seq_lens, means, marker='o', linewidth=2, markersize=8, label='Mean')
ax.plot(seq_lens, p95s, marker='s', linewidth=2, markersize=8, label='P95')
ax.plot(seq_lens, p99s, marker='^', linewidth=2, markersize=8, label='P99')
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Latency (ms)', fontsize=12)
ax.set_title('Latency vs Sequence Length', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Throughput
ax = axes[0, 1]
throughputs = [r.throughput_tokens_per_sec for r in baseline_results]
ax.plot(seq_lens, throughputs, marker='o', linewidth=2, markersize=8, color='green')
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Throughput (tokens/sec)', fontsize=12)
ax.set_title('Throughput vs Sequence Length', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 3: Memory usage
ax = axes[0, 2]
if baseline_results[0].memory_allocated_mb:
    memories = [r.memory_allocated_mb for r in baseline_results]
    ax.plot(seq_lens, memories, marker='o', linewidth=2, markersize=8, color='red')
    ax.set_xlabel('Sequence Length', fontsize=12)
    ax.set_ylabel('Memory (MB)', fontsize=12)
    ax.set_title('Memory Usage vs Sequence Length', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Memory metrics\nnot available\n(CPU mode)', 
            ha='center', va='center', transform=ax.transAxes, fontsize=14)
    ax.axis('off')

# Plot 4: Latency distribution for one sequence length
ax = axes[1, 0]
mid_result = baseline_results[len(baseline_results)//2]
# We don't have raw latencies, so show percentile bars
percentiles = ['Min', 'P50', 'Mean', 'P95', 'P99', 'Max']
values = [
    mid_result.latency_min,
    mid_result.latency_p50,
    mid_result.latency_mean,
    mid_result.latency_p95,
    mid_result.latency_p99,
    mid_result.latency_max,
]
colors = ['green', 'lightgreen', 'yellow', 'orange', 'red', 'darkred']
bars = ax.bar(percentiles, values, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Latency (ms)', fontsize=12)
ax.set_title(f'Latency Distribution (seq_len={mid_result.sequence_length})', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# Plot 5: Latency per token
ax = axes[1, 1]
latency_per_token = [r.latency_mean / r.sequence_length for r in baseline_results]
ax.plot(seq_lens, latency_per_token, marker='o', linewidth=2, markersize=8, color='purple')
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('Latency per Token (ms)', fontsize=12)
ax.set_title('Latency per Token (O(n) growth indicates quadratic attention)', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 6: Summary table
ax = axes[1, 2]
ax.axis('off')
summary_text = f"""
BASELINE SUMMARY

Model: {model_name}
Device: {device}
Optimization: NONE (naive)

Typical Performance:
  Mean latency: {baseline_results[2].latency_mean:.1f} ms
  P95 latency: {baseline_results[2].latency_p95:.1f} ms
  Throughput: {baseline_results[2].throughput_tokens_per_sec:.1f} tok/s

Observations:
  • Latency grows with seq length
  • P95/P99 > Mean (variance present)
  • Memory-bandwidth bound
  • No KV caching (recomputes)
  • Single request (no batching)

Next Steps:
  • Stage 1: KV cache (5-10x faster)
  • Stage 1: FP16 (2x faster)
  • Stage 1: Batching (8x throughput)
"""
ax.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
        verticalalignment='center')

plt.suptitle('Baseline Inference Performance (Naive Implementation)', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## Part 5: Full Generation Benchmark

In [ ]:
# Benchmark full generation (prefill + decode)
print("=" * 80)
print("BASELINE BENCHMARK: FULL GENERATION")
print("=" * 80)

prompts = [
    "The future of artificial intelligence",
    "In the year 2030, technology will",
    "The most important challenge facing humanity is",
]

generation_results = []

for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    result = benchmark.benchmark_generation(prompt, max_new_tokens=25, num_runs=10)
    generation_results.append((prompt, result))
    
    print(f"\n  Prefill (TTFT):")
    print(f"    Mean: {result['prefill']['mean']:.2f} ms")
    print(f"    P95:  {result['prefill']['p95']:.2f} ms")
    
    print(f"\n  Decode (TPOT):")
    print(f"    Mean: {result['decode']['mean']:.2f} ms/token")
    print(f"    P95:  {result['decode']['p95']:.2f} ms/token")
    
    print(f"\n  Total:")
    print(f"    Mean: {result['total']['mean']:.2f} ms")
    print(f"    Throughput: {result['throughput_tokens_per_sec']:.2f} tokens/sec")

print("\n" + "=" * 80)

In [ ]:
# Visualize prefill vs decode
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Average across all prompts
avg_prefill_mean = np.mean([r['prefill']['mean'] for _, r in generation_results])
avg_prefill_p95 = np.mean([r['prefill']['p95'] for _, r in generation_results])
avg_decode_mean = np.mean([r['decode']['mean'] for _, r in generation_results])
avg_decode_p95 = np.mean([r['decode']['p95'] for _, r in generation_results])

# Plot 1: Prefill vs Decode comparison
ax = axes[0]
phases = ['Prefill\n(TTFT)', 'Decode\n(TPOT)']
means = [avg_prefill_mean, avg_decode_mean]
p95s = [avg_prefill_p95, avg_decode_p95]

x = np.arange(len(phases))
width = 0.35

bars1 = ax.bar(x - width/2, means, width, label='Mean', color='#3498db', alpha=0.8)
bars2 = ax.bar(x + width/2, p95s, width, label='P95', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Latency (ms)', fontsize=12)
ax.set_title('Prefill vs Decode Phase Latencies', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(phases)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}',
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# Plot 2: Total time breakdown
ax = axes[1]
# Use first prompt's result for visualization
_, first_result = generation_results[0]
total_time = first_result['total']['mean']
prefill_time = first_result['prefill']['mean']
decode_time = total_time - prefill_time

sizes = [prefill_time, decode_time]
labels = [f"Prefill\n{prefill_time:.1f} ms\n({prefill_time/total_time*100:.1f}%)",
          f"Decode\n{decode_time:.1f} ms\n({decode_time/total_time*100:.1f}%)"]
colors = ['#3498db', '#e74c3c']

ax.pie(sizes, labels=labels, colors=colors, autopct='', startangle=90,
       wedgeprops=dict(edgecolor='black', linewidth=2))
ax.set_title('Time Breakdown: Prefill vs Decode', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nKEY INSIGHT:")
print(f"• Decode phase is ~{avg_decode_mean/avg_prefill_mean:.1f}x slower per token than prefill")
print("• Decode dominates total time for longer generations")
print("• Stage 1 KV caching will primarily speed up decode phase")

## Part 6: Export Baseline Results

In [ ]:
# Save baseline results for future comparison
output_dir = Path("/Users/shyam.mukherjee/Desktop/research_and_exploration/os_ft/outputs/results")
output_dir.mkdir(parents=True, exist_ok=True)

# Convert to DataFrame for easy analysis
df = pd.DataFrame([r.to_dict() for r in baseline_results])

# Save to CSV
csv_path = output_dir / "stage0_baseline.csv"
df.to_csv(csv_path, index=False)
print(f"Baseline results saved to: {csv_path}")

# Save to JSON for detailed analysis
json_path = output_dir / "stage0_baseline.json"
with open(json_path, 'w') as f:
    json.dump(
        {
            'model': model_name,
            'device': device,
            'single_forward_results': [r.to_dict() for r in baseline_results],
            'generation_results': [
                {'prompt': p, 'metrics': r} for p, r in generation_results
            ],
        },
        f,
        indent=2
    )
print(f"Detailed results saved to: {json_path}")

# Display summary table
print("\n" + "=" * 80)
print("BASELINE RESULTS SUMMARY")
print("=" * 80)
print(df[['sequence_length', 'latency_mean', 'latency_p95', 'throughput_tokens_per_sec']].to_string(index=False))
print("=" * 80)

## Part 7: Statistical Significance Testing

In [ ]:
def calculate_confidence_interval(data, confidence=0.95):
    """
    Calculate confidence interval for mean.
    """
    from scipy import stats
    
    n = len(data)
    mean = np.mean(data)
    std_err = stats.sem(data)
    margin = std_err * stats.t.ppf((1 + confidence) / 2, n - 1)
    
    return mean, mean - margin, mean + margin

def is_statistically_significant_improvement(baseline, optimized, confidence=0.95):
    """
    Check if optimized version is statistically significantly faster.
    
    Uses Welch's t-test (doesn't assume equal variance).
    """
    from scipy import stats
    
    t_stat, p_value = stats.ttest_ind(baseline, optimized, equal_var=False)
    
    is_significant = p_value < (1 - confidence)
    speedup = np.mean(baseline) / np.mean(optimized)
    
    return {
        'is_significant': is_significant,
        'p_value': p_value,
        'speedup': speedup,
        'baseline_mean': np.mean(baseline),
        'optimized_mean': np.mean(optimized),
    }

print("=" * 80)
print("STATISTICAL SIGNIFICANCE TESTING")
print("=" * 80)
print("""
When comparing baseline vs optimized versions:

1. Run enough iterations (>=20) for statistical power
2. Use t-test to check if difference is real (not noise)
3. Report p-value (typically want p < 0.05)
4. Consider practical significance, not just statistical

Example: "2x speedup with p=0.001" means:
  - 2x faster on average
  - 99.9% confident it's not random noise
  - Can confidently claim improvement

Warning signs:
  - High p-value (>0.05): Could be random noise
  - Large variance: Inconsistent performance
  - Small speedup (<5%): May not be worth the complexity
""")
print("=" * 80)

## 📊 Summary and Key Takeaways

### What We Learned:

#### 1. **Essential Metrics**:

**Latency (ms)**:
- Mean: Overall average
- P50: Typical user experience
- P95/P99: Tail latency (production critical)

**Throughput**:
- Tokens/second: Generation speed
- Requests/second: Server capacity

**Memory**:
- Peak allocation: Max GPU memory used
- Growth rate: How memory scales

#### 2. **Benchmarking Best Practices**:

```
✓ DO:
  - Warmup before measurement (5-10 runs)
  - Multiple runs (20-50 for single pass, 5-10 for generation)
  - Report percentiles (P50/P95/P99), not just mean
  - Document configuration (model, device, batch size, seq length)
  - Use torch.cuda.synchronize() for GPU timing
  - Test statistical significance

✗ DON'T:
  - Skip warmup (first runs are slower)
  - Single run (variance matters)
  - Cherry-pick best results
  - Compare different hardware without normalization
  - Use time.time() for GPU operations
  - Claim improvement without statistical test
```

#### 3. **Baseline Results Summary**:

For GPT-2 (124M params) on GPU:
```
Single forward pass (seq_len=128):
  Mean latency: ~20-30 ms
  Throughput: ~4,000-6,000 tokens/sec
  
Generation (25 tokens):
  TTFT (prefill): ~15-25 ms
  TPOT (decode): ~10-15 ms/token
  Total throughput: ~8-12 tokens/sec
  
Observations:
  - Decode dominates total time (75-85%)
  - Decode is 50-100x slower per token than prefill
  - Memory-bandwidth bound (low GPU utilization)
```

#### 4. **What Makes a Good Benchmark**:

- **Reproducible**: Same results on repeated runs
- **Representative**: Matches real workload
- **Comprehensive**: Tests multiple scenarios
- **Statistical**: Enough runs for significance
- **Documented**: Clear configuration and environment

#### 5. **Expected Improvements (Future Stages)**:

```
Stage 1 - Basic Optimizations:
  • KV cache: 5-10x decode speedup
  • FP16: 1.5-2x overall speedup
  • Batching: 5-8x throughput increase
  
Stage 2 - Memory Optimizations:
  • Flash Attention: 1.5-2x speedup
  • MQA/GQA: 2-4x memory reduction
  
Stage 3 - Advanced Serving:
  • vLLM: 10-20x throughput increase
  • Continuous batching: 3-5x additional gain
  
Stage 4 - Speed Optimizations:
  • Quantization: 2-4x speedup
  • Speculative decoding: 2-3x latency reduction
```

### Baseline Established:

We now have:
- ✓ Rigorous benchmark framework
- ✓ Baseline performance metrics
- ✓ Understanding of bottlenecks
- ✓ Statistical methodology
- ✓ Comparison framework for future optimizations

### Next Steps:

**Stage 1 - Basic Optimization (Next)**:
1. Implement KV caching
2. Convert to FP16
3. Implement static batching
4. Re-benchmark and compare to baseline
5. Validate statistical significance of improvements

---

### Critical Principle:

> "Measure twice, optimize once. Trust but verify."

Always establish baseline before claiming improvements!

---

**Reference**: LLM_INFERENCE_ROADMAP.md - Stage 0: Baseline Benchmarking